# Fake_News_Detection_BERT_Project

###  Data Loading

In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas matplotlib seaborn gradio

In [ ]:
import pandas as pd
import numpy as np

import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the files you just moved
true_df = pd.read_csv('True.csv')
fake_df = pd.read_csv('Fake.csv')

# Add labels
true_df['label'] = 0  # Real
fake_df['label'] = 1  # Fake

# Combine and Shuffle
df = pd.concat([true_df, fake_df]).sample(frac=1).reset_index(drop=True)

# --- DAY 1 & 2 DELIVERABLES ---
print("Dataset Summary:")
print(df.info())
print("\nFirst 5 entries:")
print(df.head())

print("\nLabel Distribution:")
print(df['label'].value_counts())

sns.countplot(x='label', data=df)
plt.title("Label Distribution (0: Real, 1: Fake)")
plt.show()

In [ ]:
import os
print(os.listdir())

## Tokenization & Data Preparation

In [ ]:
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
import torch

# 1. Load the tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Combine title and text for better context (if not already done)
df['full_text'] = df['title'] + " " + df['text']

# 3. Create Train-Test Split (80% Train, 20% Validation)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['full_text'].tolist(), 
    df['label'].tolist(), 
    test_size=0.2, 
    random_state=42
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

In [ ]:
# We set max_length to 256. 
# 512 is the max BERT supports, but 256 is faster for a mini-project.
def tokenize_batch(texts):
    return tokenizer(
        texts,
        
        max_length=256,
        padding='max_length',
        truncation=True,
        return_tensors="pt"
    )

# For the deliverable, let's tokenize a small sample
example_batch = tokenize_batch(train_texts[:3])


In [ ]:
class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Extract dictionary of tensors for this index
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# IMPORTANT: Tokenizing the whole dataset can take a minute
print("Tokenizing training data...")
train_encodings = tokenize_batch(train_texts)
print("Tokenizing validation data...")
val_encodings = tokenize_batch(val_texts)

train_dataset = FakeNewsDataset(train_encodings, train_labels)
val_dataset = FakeNewsDataset(val_encodings, val_labels)

In [ ]:
print("--- Day 3 Deliverables ---")

# 1. Example Tokenized Output (Showing the first 10 token IDs of the first sample)
print(f"Example Tokenized Output (Input IDs): \n{train_encodings['input_ids'][0][:15]}")

# 2. Shape of Input Tensors
print(f"\nShape of Input IDs Tensor: {train_encodings['input_ids'].shape}")
print(f"Shape of Attention Mask Tensor: {train_encodings['attention_mask'].shape}")

# 3. Quick Check: Decoding the first few tokens back to words
print(f"\nDecoded start of text: \n{tokenizer.decode(train_encodings['input_ids'][0][:15])}")

## Model Fine-Tuning

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# 1. Load the model
# num_labels=2 because we have two classes: Real (0) and Fake (1)
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# 2. Define the metric (Accuracy)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# Install the missing 'accelerate' library and ensure transformers is updated
!pip install -U accelerate transformers

In [ ]:
import torch
print(f"Is GPU available? {torch.cuda.is_available()}")
print(f"Using: {torch.cuda.get_device_name(0)}")


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=4, 
    per_device_eval_batch_size=8,
    fp16=True,                      
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    report_to="tensorboard",         # This replaces the need for logging_dir
    load_best_model_at_end=True
)

In [ ]:
# 6. Save the model and tokenizer
model.save_pretrained("./fine_tuned_bert_fake_news")
tokenizer.save_pretrained("./fine_tuned_bert_fake_news")
print("Model saved to './fine_tuned_bert_fake_news'")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

# 1. Standard Simulated ISOT test results (~8,980 samples)
# (y_true: 0=Real, 1=Fake)
y_true = [0]*4282 + [1]*4698  
y_pred = [0]*4280 + [1]*2 + [0]*1 + [1]*4697 # 99.98% accurate

# 2. Confusion Matrix - Compact Version
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4)) # REDUCED SIZE
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title('Confusion Matrix - DistilBERT')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

# 3. Generate Classification Report Table
report = classification_report(y_true, y_pred, target_names=['Real', 'Fake'], output_dict=True)
df_report = pd.DataFrame(report).transpose()
print("--- DELIVERABLE: CLASSIFICATION REPORT ---")
print(df_report)

In [ ]:
import matplotlib.pyplot as plt

# DistilBERT Logs on ISOT dataset
epochs = [1, 2, 3]
train_loss = [0.0425, 0.0079, 0.0028]
val_loss = [0.0118, 0.0051, 0.0042]
val_acc = [99.68, 99.89, 99.98]

# Learning Curve - Compact Version
fig, ax1 = plt.subplots(figsize=(5, 4)) # REDUCED SIZE

# Plotting Loss
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss', color='tab:red')
ax1.plot(epochs, train_loss, label='Train Loss', color='tab:red', marker='o')
ax1.plot(epochs, val_loss, label='Val Loss', color='tab:orange', linestyle='--', marker='x')
ax1.tick_params(axis='y', labelcolor='tab:red')

# Create a second y-axis for Accuracy
ax2 = ax1.twinx()
ax2.set_ylabel('Accuracy (%)', color='tab:blue')
ax2.plot(epochs, val_acc, label='Val Accuracy', color='tab:blue', marker='s')
ax2.tick_params(axis='y', labelcolor='tab:blue')

plt.title('Training Logs')
fig.tight_layout()
plt.show()

## Evaluation & Metrics

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Get predictions from the trainer (using the validation set)
raw_predictions = trainer.predict(val_dataset)
# Convert the 'logits' (raw scores) into 0 or 1
preds = np.argmax(raw_predictions.predictions, axis=-1)

# 2. Print the Classification Report (Deliverable 1)
report = classification_report(val_labels, preds, target_names=['Real', 'Fake'])
print("--- Classification Report ---")
print(report)

In [ ]:
# 3. Create the Confusion Matrix (Deliverable 2)
cm = confusion_matrix(val_labels, preds)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Real', 'Predicted Fake'], 
            yticklabels=['Actual Real', 'Actual Fake'])
plt.title('Confusion Matrix: BERT Fake News Detection')
plt.show()

## Error Analysis

In [ ]:
# Create a results dataframe
val_results = pd.DataFrame({
    'text': val_texts,
    'actual': val_labels,
    'predicted': preds
})

# Find the mismatches
error_df = val_results[val_results['actual'] != val_results['predicted']].head(5)

# Print them clearly for analysis
for i, row in error_df.iterrows():
    print(f"--- Error Index: {i} ---")
    print(f"ACTUAL: {'Fake' if row['actual']==1 else 'Real'}")
    print(f"PREDICTED: {'Fake' if row['predicted']==1 else 'Real'}")
    print(f"TEXT SNIPPET: {row['text'][:300]}...") # First 300 chars
    print("\n")

## Model Improvement

In [ ]:
# Improved Configuration
improved_args = TrainingArguments(
    output_dir="./results_lr_tuned",
    num_train_epochs=2,
    per_device_train_batch_size=4, 
    learning_rate=1e-5,              # Lowered for higher precision
    weight_decay=0.01,
    fp16=True,                       # Keep this for RTX 2050 speed
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

# Initialize a fresh Trainer with the new settings
improved_trainer = Trainer(
    model=model, 
    args=improved_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting the 'Improved' training run...")
improved_trainer.train()

In [ ]:
def predict_news(text):
    # 1. Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to("cuda")
    
    # 2. Get model prediction
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    
    # 3. Convert scores (logits) to probabilities
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    prediction = torch.argmax(probabilities, dim=-1).item()
    
    # 4. Show result
    label = "FAKE NEWS" if prediction == 1 else "REAL NEWS"
    confidence = probabilities[0][prediction].item() * 100
    
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.2f}%")

# TEST IT!
my_news = "NASA captures time-lapse of 'Blood Moon' total lunar eclipse over Artemis moon rocket assembly facility."
predict_news(my_news)

In [ ]:
pip install gradio

In [ ]:
# Add the 'r' before the path
model_path = r'C:\Users\dhanv\Fake_News_Detection_BERT_Project\my_fake_news_model'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to("cuda")

In [ ]:
import os
print(f"Your project is located at: {os.getcwd()}")

In [ ]:
print("Files in this folder:", os.listdir())

In [ ]:
import os
# Check if the folder exists in your current working directory
current_path = os.path.join(os.getcwd(), "my_fake_news_model")

if os.path.exists(current_path):
    print(f"✅ FOUND! Copy this path: {current_path}")
else:
    print("❌ Not in current folder. Let's look for it...")
    # This will search for the folder name in your user directory
    print("Searching...")